In [ ]:
# =========================
# KERAS(Hybrid + Ablation)
# TF 2.20 + Keras3 compatible, SINGLE CELL
# Models: EfficientNetV2B0, MobileNetV3Small
# Scenarios:
#   (A) HYBRID: PlantVillage pretrain -> PlantDoc fine-tune
#   (B) NO-PRETRAIN ABLATION: ImageNet -> PlantDoc only
# Outputs:
#   - Loss/Acc plots
#   - Confusion matrices (raw+norm)
#   - ROC micro + AUC
#   - Params / GFLOPs / Latency
#   - Checkpoints, histories, summary.csv/json
# =========================

from google.colab import drive
drive.mount('/content/drive')

!pip -q install -U scikit-learn tqdm matplotlib

import os, time, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import confusion_matrix, roc_curve, auc, f1_score, accuracy_score
from sklearn.preprocessing import label_binarize

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

try:
    tf.config.experimental.set_synchronous_execution(True)
    print("Synchronous execution: ON")
except Exception as e:
    print("Synchronous execution could not be set:", e)

# ---------- CONFIG ----------
class CFG:
    seed = 42
    img_size = 224
    batch_size = 64

    # Max epochs 30 (EarlyStopping daha erken durdurabilir)
    pv_epochs_head = 5
    pv_epochs_ft   = 20
    pv_lr_head     = 3e-4
    pv_lr_ft       = 1e-5

    pd_epochs_head = 5
    pd_epochs_ft   = 25
    pd_lr_head     = 3e-4
    pd_lr_ft       = 1e-5

    # Ablation (No-Pretrain) - PlantDoc only
    ab_epochs_head = 5
    ab_epochs_ft   = 20
    ab_lr_head     = 3e-4
    ab_lr_ft       = 1e-5

    weight_decay = 1e-4
    label_smoothing = 0.1
    val_size = 0.15

    early_stop_patience = 6
    lr_plateau_patience = 2
    lr_plateau_factor = 0.5
    min_lr = 1e-6

    PLANTDOC_BASE  = "/content/drive/MyDrive/plant/plantdoc"
    PLANTVILLAGE_BASE = "/content/drive/MyDrive/plant/plantvillage"

    OUT_BASE = "/content/drive/MyDrive/hybrid_plant_results2"
    os.makedirs(OUT_BASE, exist_ok=True)

tf.keras.utils.set_random_seed(CFG.seed)
np.random.seed(CFG.seed)
random.seed(CFG.seed)

PLANTDOC_TRAIN = os.path.join(CFG.PLANTDOC_BASE, "train")
PLANTDOC_TEST  = os.path.join(CFG.PLANTDOC_BASE, "test")

# ---------- HELPERS ----------
def safe_name(s: str) -> str:
    return s.replace("/", "_").replace(":", "_").replace(" ", "_")

def list_subdirs(path):
    return [d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))]

def is_class_root(path, min_classes=10, min_images_sample=50):
    if not os.path.isdir(path):
        return False, (0,0)
    subdirs = list_subdirs(path)
    if len(subdirs) < min_classes:
        return False, (len(subdirs), 0)
    exts = (".jpg",".jpeg",".png",".bmp",".webp")
    img_count = 0
    for d in subdirs[:min(60, len(subdirs))]:
        dp = os.path.join(path, d)
        try:
            for fn in os.listdir(dp)[:2000]:
                if fn.lower().endswith(exts):
                    img_count += 1
        except:
            pass
    return (img_count >= min_images_sample), (len(subdirs), img_count)

def find_best_class_root(base_path, depth=6):
    candidates = []
    for root, dirs, files in os.walk(base_path):
        rel_depth = root[len(base_path):].count(os.sep)
        if rel_depth > depth:
            dirs[:] = []
            continue
        ok, (ncls, nimg) = is_class_root(root)
        if ok:
            candidates.append((root, ncls, nimg))
    candidates.sort(key=lambda x: (x[1], x[2]), reverse=True)
    return candidates

def ensure_same_class_list(train_dir, test_dir):
    tr = sorted(list_subdirs(train_dir))
    te = sorted(list_subdirs(test_dir))
    if tr != te:
        raise ValueError(
            "Train ve test class listesi aynı değil!\n"
            f"Train classes({len(tr)}): {tr[:10]}...\n"
            f"Test  classes({len(te)}): {te[:10]}...\n"
            "Klasör isimleri birebir aynı olmalı."
        )
    return tr

# label_mode='categorical' => label_smoothing Keras3 uyumlu
def make_datasets_from_directory(root_dir, val_split=0.15, img_size=224, batch=64, seed=42, shuffle=True):
    train_ds = tf.keras.utils.image_dataset_from_directory(
        root_dir, labels="inferred", label_mode="categorical",
        image_size=(img_size, img_size),
        batch_size=batch, shuffle=shuffle, seed=seed,
        validation_split=val_split, subset="training"
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        root_dir, labels="inferred", label_mode="categorical",
        image_size=(img_size, img_size),
        batch_size=batch, shuffle=shuffle, seed=seed,
        validation_split=val_split, subset="validation"
    )
    return train_ds, val_ds

def make_test_dataset(root_dir, img_size=224, batch=64):
    return tf.keras.utils.image_dataset_from_directory(
        root_dir, labels="inferred", label_mode="categorical",
        image_size=(img_size, img_size),
        batch_size=batch, shuffle=False
    )

AUTOTUNE = tf.data.AUTOTUNE

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
], name="augment")

def prepare(ds, training=False):
    if training:
        ds = ds.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
        ds = ds.shuffle(2048, seed=CFG.seed, reshuffle_each_iteration=True)
    return ds.cache().prefetch(AUTOTUNE)

def save_history_plots(hist_df, out_dir, title_prefix=""):
    plt.figure()
    plt.plot(hist_df["loss"], label="train_loss")
    plt.plot(hist_df["val_loss"], label="val_loss")
    plt.legend()
    plt.title(f"{title_prefix} Loss")
    plt.xlabel("epoch"); plt.ylabel("loss")
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{title_prefix}_loss.png"), dpi=200)
    plt.close()

    if "accuracy" in hist_df.columns and "val_accuracy" in hist_df.columns:
        plt.figure()
        plt.plot(hist_df["accuracy"], label="train_acc")
        plt.plot(hist_df["val_accuracy"], label="val_acc")
        plt.legend()
        plt.title(f"{title_prefix} Accuracy")
        plt.xlabel("epoch"); plt.ylabel("acc")
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"{title_prefix}_acc.png"), dpi=200)
        plt.close()

def save_confusion_with_numbers(y_true, y_pred, out_path, title="", normalize=False):
    cm = confusion_matrix(y_true, y_pred)
    cm_disp = (cm.astype(np.float64) / np.maximum(cm.sum(axis=1, keepdims=True), 1)) if normalize else cm

    plt.figure(figsize=(10, 10))
    plt.imshow(cm_disp, interpolation="nearest")
    plt.title(title + (" (normalized)" if normalize else " (raw)"))
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.colorbar()

    n = cm_disp.shape[0]
    for i in range(n):
        for j in range(n):
            val = cm_disp[i, j]
            txt = f"{val:.2f}" if normalize else f"{int(val)}"
            plt.text(j, i, txt, ha="center", va="center", fontsize=7)

    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()

def save_roc_micro(y_true, prob, n_classes, out_path, title=""):
    Y = label_binarize(y_true, classes=np.arange(n_classes))
    fpr, tpr, _ = roc_curve(Y.ravel(), prob.ravel())
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, label=f"micro-average AUC={roc_auc:.4f}")
    plt.plot([0,1],[0,1], linestyle="--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"{title} ROC (micro-average)")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    plt.close()
    return float(roc_auc)

def compute_latency_ms(model, img_size=224, warmup=20, iters=100):
    x = tf.random.uniform([1, img_size, img_size, 3], dtype=tf.float32)

    @tf.function
    def _infer(x):
        return model(x, training=False)

    for _ in range(warmup):
        _ = _infer(x)

    t0 = time.time()
    for _ in range(iters):
        _ = _infer(x)
    t1 = time.time()
    return (t1 - t0) * 1000.0 / iters

def compute_flops(model, img_size=224):
    try:
        from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

        @tf.function
        def _f(x):
            return model(x, training=False)

        concrete = _f.get_concrete_function(tf.TensorSpec([1, img_size, img_size, 3], tf.float32))
        frozen_func = convert_variables_to_constants_v2(concrete)
        graph_def = frozen_func.graph.as_graph_def()

        with tf.Graph().as_default() as graph:
            tf.import_graph_def(graph_def, name="")
            run_meta = tf.compat.v1.RunMetadata()
            opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
            prof = tf.compat.v1.profiler.profile(graph=graph, run_meta=run_meta, cmd="op", options=opts)
            if prof is None:
                return None
            return float(prof.total_float_ops) / 1e9
    except Exception as e:
        print("FLOPs calc failed:", e)
        return None

# ---------- MODEL BUILDERS ----------
def build_backbone_and_preprocess(model_key, img_size=224):
    if model_key == "EfficientNetV2B0":
        preprocess = tf.keras.applications.efficientnet_v2.preprocess_input
        backbone = tf.keras.applications.EfficientNetV2B0(
            include_top=False, weights="imagenet",
            input_shape=(img_size, img_size, 3), pooling="avg"
        )
        return backbone, preprocess

    if model_key == "MobileNetV3Small":
        preprocess = tf.keras.applications.mobilenet_v3.preprocess_input
        backbone = tf.keras.applications.MobileNetV3Small(
            include_top=False, weights="imagenet",
            input_shape=(img_size, img_size, 3), pooling="avg"
        )
        return backbone, preprocess

    raise ValueError("Unknown model_key")

def build_classifier_model(model_key, num_classes, img_size=224, dropout=0.2):
    backbone, preprocess = build_backbone_and_preprocess(model_key, img_size=img_size)
    inp = keras.Input(shape=(img_size, img_size, 3))
    x = preprocess(inp)
    x = backbone(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    model = keras.Model(inp, out, name=f"{model_key}_clf")
    return model, backbone

def compile_model(model, lr):
    opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=CFG.weight_decay)
    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=CFG.label_smoothing)
    model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])
    return model

def callbacks_for(out_dir, tag):
    return [
        keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(out_dir, f"{tag}_best.keras"),
            monitor="val_loss", mode="min",
            save_best_only=True, save_weights_only=False
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss", mode="min",
            patience=CFG.early_stop_patience,
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", mode="min",
            factor=CFG.lr_plateau_factor,
            patience=CFG.lr_plateau_patience,
            min_lr=CFG.min_lr
        )
    ]

# ---------- EVAL ----------
def predict_probs_and_labels(model, ds):
    probs = model.predict(ds, verbose=0)
    y_true_oh = np.concatenate([y.numpy() for _, y in ds], axis=0)
    y_true = y_true_oh.argmax(axis=1)
    y_pred = probs.argmax(axis=1)
    return probs, y_true, y_pred

def evaluate_and_save(model, ds, num_classes, out_dir, tag):
    probs, y_true, y_pred = predict_probs_and_labels(model, ds)
    acc = float(accuracy_score(y_true, y_pred))
    macro_f1 = float(f1_score(y_true, y_pred, average="macro"))
    micro_auc = save_roc_micro(y_true, probs, num_classes,
                               os.path.join(out_dir, f"{tag}_roc_micro.png"), title=tag)

    save_confusion_with_numbers(y_true, y_pred, os.path.join(out_dir, f"{tag}_cm_raw.png"),
                                title=tag, normalize=False)
    save_confusion_with_numbers(y_true, y_pred, os.path.join(out_dir, f"{tag}_cm_norm.png"),
                                title=tag, normalize=True)
    return acc, macro_f1, micro_auc

def get_complexity(model):
    params_m = model.count_params() / 1e6
    flops_g = compute_flops(model, img_size=CFG.img_size)
    latency_ms = compute_latency_ms(model, img_size=CFG.img_size)
    return float(params_m), (float(flops_g) if flops_g is not None else None), float(latency_ms)

def find_backbone_layer(m):
    for lyr in m.layers:
        if isinstance(lyr, tf.keras.Model) and len(lyr.weights) > 0:
            return lyr
    return None

# ---------- TRAINERS ----------
def train_stage(model, backbone, train_ds, val_ds, out_dir, tag, epochs_head, epochs_ft, lr_head, lr_ft):
    train_p = prepare(train_ds, training=True)
    val_p   = prepare(val_ds, training=False)
    cbs = callbacks_for(out_dir, tag)

    # head
    backbone.trainable = False
    compile_model(model, lr_head)
    h1 = model.fit(train_p, validation_data=val_p, epochs=epochs_head, callbacks=cbs, verbose=1)

    # fine-tune
    backbone.trainable = True
    compile_model(model, lr_ft)
    h2 = model.fit(train_p, validation_data=val_p, epochs=epochs_ft, callbacks=cbs, verbose=1)

    hist_df = pd.DataFrame({k: h1.history.get(k, []) + h2.history.get(k, [])
                            for k in set(h1.history.keys()).union(h2.history.keys())})
    hist_df.to_csv(os.path.join(out_dir, f"{tag}_history.csv"), index=False)
    save_history_plots(hist_df, out_dir, title_prefix=tag)
    return model

def run_hybrid(model_key, pv_root, pd_train_dir, pd_test_dir):
    run_tag = safe_name(model_key)
    out_dir = os.path.join(CFG.OUT_BASE, run_tag, "HYBRID")
    os.makedirs(out_dir, exist_ok=True)

    # PV data
    pv_train_ds, pv_val_ds = make_datasets_from_directory(
        pv_root, val_split=CFG.val_size, img_size=CFG.img_size, batch=CFG.batch_size, seed=CFG.seed, shuffle=True
    )
    pv_num_classes = len(pv_train_ds.class_names)
    print(f"\n[HYBRID] {model_key} | PV classes={pv_num_classes} | root={pv_root}")

    pv_model, pv_backbone = build_classifier_model(model_key, pv_num_classes, img_size=CFG.img_size, dropout=0.2)

    params_m, flops_g, latency_ms = get_complexity(pv_model)
    with open(os.path.join(out_dir, f"{run_tag}_complexity.json"), "w") as f:
        json.dump({"params_m": params_m, "gflops": flops_g, "latency_ms_batch1": latency_ms}, f, indent=2)

    # Train on PV
    pv_model = train_stage(
        pv_model, pv_backbone, pv_train_ds, pv_val_ds, out_dir, f"{run_tag}_PV",
        CFG.pv_epochs_head, CFG.pv_epochs_ft, CFG.pv_lr_head, CFG.pv_lr_ft
    )
    pv_final = os.path.join(out_dir, f"{run_tag}_PV_final.keras")
    pv_model.save(pv_final)

    # PD data
    pd_classes = ensure_same_class_list(pd_train_dir, pd_test_dir)
    pd_num_classes = len(pd_classes)

    pd_train_ds, pd_val_ds = make_datasets_from_directory(
        pd_train_dir, val_split=CFG.val_size, img_size=CFG.img_size, batch=CFG.batch_size, seed=CFG.seed, shuffle=True
    )
    pd_test_ds = make_test_dataset(pd_test_dir, img_size=CFG.img_size, batch=CFG.batch_size)
    pd_test_p  = pd_test_ds.cache().prefetch(AUTOTUNE)

    # Build PD model and transfer backbone weights from PV
    pv_loaded = keras.models.load_model(pv_final)
    pd_model, _ = build_classifier_model(model_key, pd_num_classes, img_size=CFG.img_size, dropout=0.25)

    src_bb = find_backbone_layer(pv_loaded)
    dst_bb = find_backbone_layer(pd_model)
    if src_bb is None or dst_bb is None:
        raise RuntimeError("Backbone bulunamadı (weight transfer).")
    dst_bb.set_weights(src_bb.get_weights())
    del pv_loaded

    # Train on PD
    pd_model = train_stage(
        pd_model, dst_bb, pd_train_ds, pd_val_ds, out_dir, f"{run_tag}_PD",
        CFG.pd_epochs_head, CFG.pd_epochs_ft, CFG.pd_lr_head, CFG.pd_lr_ft
    )
    pd_final = os.path.join(out_dir, f"{run_tag}_PD_final.keras")
    pd_model.save(pd_final)

    # Evaluate PD test
    acc, macro_f1, micro_auc = evaluate_and_save(pd_model, pd_test_p, pd_num_classes, out_dir, f"{run_tag}_PD_TEST")

    res = {
        "model": model_key,
        "scenario": "HYBRID_PV_to_PD",
        "params_m": params_m,
        "gflops": flops_g,
        "latency_ms_batch1": latency_ms,
        "pv_root": pv_root,
        "pv_num_classes": int(pv_num_classes),
        "pd_num_classes": int(pd_num_classes),
        "pv_final": pv_final,
        "pd_final": pd_final,
        "plantdoc_test_acc": acc,
        "plantdoc_test_macro_f1": macro_f1,
        "plantdoc_test_micro_auc": micro_auc,
        "out_dir": out_dir
    }
    with open(os.path.join(out_dir, f"{run_tag}_result.json"), "w") as f:
        json.dump(res, f, indent=2)
    return res

def run_no_pretrain_ablation(model_key, pd_train_dir, pd_test_dir):
    run_tag = safe_name(model_key)
    out_dir = os.path.join(CFG.OUT_BASE, run_tag, "NO_PRETRAIN")
    os.makedirs(out_dir, exist_ok=True)

    pd_classes = ensure_same_class_list(pd_train_dir, pd_test_dir)
    pd_num_classes = len(pd_classes)

    pd_train_ds, pd_val_ds = make_datasets_from_directory(
        pd_train_dir, val_split=CFG.val_size, img_size=CFG.img_size, batch=CFG.batch_size, seed=CFG.seed, shuffle=True
    )
    pd_test_ds = make_test_dataset(pd_test_dir, img_size=CFG.img_size, batch=CFG.batch_size)
    pd_test_p  = pd_test_ds.cache().prefetch(AUTOTUNE)

    print(f"\n[ABLATION NO-PRETRAIN] {model_key} | PD classes={pd_num_classes}")

    model, backbone = build_classifier_model(model_key, pd_num_classes, img_size=CFG.img_size, dropout=0.25)

    params_m, flops_g, latency_ms = get_complexity(model)
    with open(os.path.join(out_dir, f"{run_tag}_complexity.json"), "w") as f:
        json.dump({"params_m": params_m, "gflops": flops_g, "latency_ms_batch1": latency_ms}, f, indent=2)

    # Train only on PlantDoc (ImageNet -> PlantDoc)
    model = train_stage(
        model, backbone, pd_train_ds, pd_val_ds, out_dir, f"{run_tag}_PD",
        CFG.ab_epochs_head, CFG.ab_epochs_ft, CFG.ab_lr_head, CFG.ab_lr_ft
    )
    pd_final = os.path.join(out_dir, f"{run_tag}_PD_final.keras")
    model.save(pd_final)

    # Evaluate PD test
    acc, macro_f1, micro_auc = evaluate_and_save(model, pd_test_p, pd_num_classes, out_dir, f"{run_tag}_PD_TEST")

    res = {
        "model": model_key,
        "scenario": "NO_PRETRAIN_ImageNet_to_PD",
        "params_m": params_m,
        "gflops": flops_g,
        "latency_ms_batch1": latency_ms,
        "pd_num_classes": int(pd_num_classes),
        "pd_final": pd_final,
        "plantdoc_test_acc": acc,
        "plantdoc_test_macro_f1": macro_f1,
        "plantdoc_test_micro_auc": micro_auc,
        "out_dir": out_dir
    }
    with open(os.path.join(out_dir, f"{run_tag}_result.json"), "w") as f:
        json.dump(res, f, indent=2)
    return res

# ---------- FIND PLANTVILLAGE ROOT ----------
print("\nSearching PlantVillage class-root under:", CFG.PLANTVILLAGE_BASE)
cands = find_best_class_root(CFG.PLANTVILLAGE_BASE, depth=6)
print("Candidates found:", len(cands))
for i,(p,nc,ni) in enumerate(cands[:10]):
    print(f"{i:02d} | classes={nc:3d} | sample_imgs={ni:5d} | {p}")
if not cands:
    raise RuntimeError("PlantVillage class-root bulunamadı. PLANTVILLAGE_BASE yolunu kontrol et.")
BEST_PLANTVILLAGE_ROOT = cands[0][0]
print("\nBEST_PLANTVILLAGE_ROOT =", BEST_PLANTVILLAGE_ROOT)

# PlantDoc sanity
_ = ensure_same_class_list(PLANTDOC_TRAIN, PLANTDOC_TEST)
print("[PlantDoc] OK:", PLANTDOC_TRAIN, PLANTDOC_TEST)

# ---------- RUN BOTH SCENARIOS ----------
MODELS = ["EfficientNetV2B0", "MobileNetV3Small"]

results = []
for mk in MODELS:
    try:
        # Ablation first (quick feedback)
        results.append(run_no_pretrain_ablation(mk, PLANTDOC_TRAIN, PLANTDOC_TEST))
        # Hybrid
        results.append(run_hybrid(mk, BEST_PLANTVILLAGE_ROOT, PLANTDOC_TRAIN, PLANTDOC_TEST))
    except Exception as e:
        print(f"❌ Failed: {mk} | {e}")
        results.append({"model": mk, "error": str(e)})

# ---------- SAVE GLOBAL SUMMARY ----------
df = pd.DataFrame(results)
summary_csv = os.path.join(CFG.OUT_BASE, "summary.csv")
summary_json = os.path.join(CFG.OUT_BASE, "summary.json")
df.to_csv(summary_csv, index=False)
with open(summary_json, "w") as f:
    json.dump(results, f, indent=2)

print("\n" + "="*90)
print("SUMMARY SAVED")
print("CSV :", summary_csv)
print("JSON:", summary_json)

# Best by macro-F1
df_ok = df[df.get("error").isna()] if "error" in df.columns else df
if len(df_ok) and "plantdoc_test_macro_f1" in df_ok.columns:
    best = df_ok.sort_values("plantdoc_test_macro_f1", ascending=False).iloc[0]
    print("\nBest by PlantDoc test macro-F1:")
    print(best[["model","scenario","plantdoc_test_acc","plantdoc_test_macro_f1","plantdoc_test_micro_auc",
                "params_m","gflops","latency_ms_batch1","out_dir"]])
else:
    print("\nNo successful runs.")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 140.1 MB/s eta 0:00:00
TF: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Synchronous execution: ON

Searching PlantVillage class-root under: /content/drive/MyDrive/plant/plantvillage
Candidates found: 2
00 | classes= 16 | sample_imgs=19302 | /content/drive/MyDrive/plant/plantvillage
01 | classes= 15 | sample_imgs=19302 | /content/drive/MyDrive/plant/plantvillage/PlantVillage

BEST_PLANTVILLAGE_ROOT = /content/drive/MyDrive/plant/plantvillage
[PlantDoc] OK: /content/drive/MyDrive/plant/plantdoc/train /content/drive/MyDrive/plant/plantdoc/test
Found 2336 files belonging to 28 classes.
Using 1986 files for training.
Found 2336 files belonging to 28 classes.
Using 350 files for validation.
Found 238 files belonging to 28 c

Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


Epoch 1/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 168s 2s/step - accuracy: 0.0517 - loss: 3.3678 - val_accuracy: 0.1429 - val_loss: 3.0245 - learning_rate: 3.0000e-04
Epoch 2/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - accuracy: 0.1503 - loss: 3.0415 - val_accuracy: 0.2429 - val_loss: 2.8000 - learning_rate: 3.0000e-04
Epoch 3/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.2355 - loss: 2.8348 - val_accuracy: 0.2914 - val_loss: 2.6361 - learning_rate: 3.0000e-04
Epoch 4/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.2997 - loss: 2.6676 - val_accuracy: 0.3486 - val_loss: 2.5088 - learning_rate: 3.0000e-04
Epoch 5/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.3453 - loss: 2.5444 - val_accuracy: 0.3800 - val_loss: 2.4079 - learning_rate: 3.0000e-04
Epoch 1/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 142s 2s/step - accuracy: 0.2049 - loss: 3.0012 - val_accuracy: 0.2057 - val_loss: 2.9672 - learning_rate: 1.0000e-05
Epoch 2/20
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - accuracy: 0.2458 - los

Found 41276 files belonging to 16 classes.
Using 35085 files for training.
Found 41276 files belonging to 16 classes.
Using 6191 files for validation.

[HYBRID] MobileNetV3Small | PV classes=16 | root=/content/drive/MyDrive/plant/plantvillage
Epoch 1/5
549/549 ━━━━━━━━━━━━━━━━━━━━ 128s 78ms/step - accuracy: 0.3956 - loss: 2.2433 - val_accuracy: 0.4889 - val_loss: 1.5812 - learning_rate: 3.0000e-04
Epoch 2/5
549/549 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.4932 - loss: 1.6049 - val_accuracy: 0.4849 - val_loss: 1.4971 - learning_rate: 3.0000e-04
Epoch 3/5
549/549 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.4949 - loss: 1.5203 - val_accuracy: 0.4859 - val_loss: 1.4620 - learning_rate: 3.0000e-04
Epoch 4/5
549/549 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.4974 - loss: 1.4808 - val_accuracy: 0.4868 - val_loss: 1.4449 - learning_rate: 3.0000e-04
Epoch 5/5
549/549 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - accuracy: 0.5021 - loss: 1.4553 - val_accuracy: 0.4870 - val_loss: 1.4332 - l

This code trains and compares two lightweight models (EfficientNetV2B0 and MobileNetV3Small) in two different scenarios: (1) In the Ablation/No-Pretrain part, the models are trained on PlantDoc (with train/val split) starting with only ImageNet weights, and accuracy, macro-F1, and micro-AUC are calculated on the test set; (2) In the Hybrid part, the same model is first pre-trained (head training + fine-tune) on the class root automatically found from the PlantVillage folder structure, then the learned backbone weights are transferred to PlantDoc and head + fine-tune is performed again, and then the same metrics are calculated in the PlantDoc test. For each run, loss/accuracy curves (png), confusion matrix (raw and normalized png), ROC micro curve (png), checkpoint/best model and final model (keras), history.csv, and the number of parameters, GFLOPs, and single image latency of the model are saved as JSON. Finally, the results of all the runs are summarized in summary.csv and summary.json files, and the "best macro-F1" result is printed to the screen.